# §3 Structure: the recursive-descent parser

*Reads `calc/nodes.py` and `calc/parser.py`. Refines ⟨Turn the tokens into a syntax tree⟩.*

The parser is the heart of `calc`. It takes the flat token list from §2 and discovers its structure, producing an **abstract syntax tree** whose *shape* captures precedence and grouping. Once the tree exists, evaluation (§4) is almost trivial — so the cleverness of the whole program lives here.

## §3.1 What the parser produces: the AST

Before reading how the tree is built, meet the tree itself. Three node kinds suffice for arithmetic: a literal `Num`, a binary `BinOp`, and a `Unary` negation.

📄 [calc/nodes.py lines 13–31](../sample_repo/calc/nodes.py)

```python
# calc/nodes.py : lines 13-31
@dataclass
class Num:
    value: float


@dataclass
class BinOp:
    op: str  # one of: + - * /
    left: "Node"
    right: "Node"


@dataclass
class Unary:
    op: str  # currently only '-'
    operand: "Node"


Node = Union[Num, BinOp, Unary]
```

The comment in the source states the key idea: **the tree's shape encodes precedence and associativity.** `1 + 2 * 3` parses to `BinOp('+', Num(1), BinOp('*', Num(2), Num(3)))` — the multiplication is *nested deeper*, so the evaluator reaches it first, with no precedence logic of its own. The parser's job is to build exactly the right shape.

## §3.2 The grammar, and a method per rule

The parser is *recursive descent*: each grammar rule becomes one method, and rules that should bind tighter are nested deeper in the call chain. The grammar, lifted from the module's own docstring:

📄 [calc/parser.py lines 6–8](../sample_repo/calc/parser.py)

```python
# calc/parser.py : lines 6-8
    expression := term   (('+' | '-') term)*       # lowest precedence
    term       := factor (('*' | '/') factor)*      # binds tighter
    factor     := NUMBER | '(' expression ')' | '-' factor   # tightest
```

Read it top to bottom as *loosest to tightest*: an `expression` is a sum of `term`s; a `term` is a product of `factor`s; a `factor` is the atom — a number, a parenthesized expression, or a negation. Because `term` sits *inside* `expression`, multiplication is resolved before addition. Precedence is encoded in the call graph, not a table.

## §3.3 The cursor

Like the tokenizer, the parser walks its input with a cursor, exposed through three tiny helpers. `expect` is `advance` with an assertion — it is how the grammar's literal tokens (like a closing `)`) are consumed and checked at once.

📄 [calc/parser.py lines 25–36](../sample_repo/calc/parser.py)

```python
# calc/parser.py : lines 25-36
    def peek(self) -> Token:
        return self.tokens[self.pos]

    def advance(self) -> Token:
        tok = self.tokens[self.pos]
        self.pos += 1
        return tok

    def expect(self, kind: str) -> Token:
        if self.peek().kind != kind:
            raise SyntaxError(f"expected {kind}, got {self.peek().kind}")
        return self.advance()
```

## §3.4 Sums and products — and where associativity comes from

`expression` and `term` are the same loop at two precedence levels. Read `expression`: parse one `term`, then *while* the next token is `+` or `-`, fold another `term` onto the right:

📄 [calc/parser.py lines 43–55](../sample_repo/calc/parser.py)

```python
# calc/parser.py : lines 43-55
    def expression(self) -> Node:
        node = self.term()
        while self.peek().kind in (PLUS, MINUS):
            op = self.advance().value
            node = BinOp(op, node, self.term())
        return node

    def term(self) -> Node:
        node = self.factor()
        while self.peek().kind in (STAR, SLASH):
            op = self.advance().value
            node = BinOp(op, node, self.factor())
        return node
```

The `while` loop is where **left-associativity** is born. For `8 - 2 - 1` it builds `BinOp('-', BinOp('-', 8, 2), 1)` — the left operand grows as we iterate, so subtraction groups left-to-right and the answer is `5`, not `7`. Had we written this with recursion on the right instead of a loop, the grouping — and the answer — would flip. A one-line structural choice with an arithmetic consequence: exactly the kind of *tough place* a literate reading must not gloss over.

## §3.5 Atoms, parentheses, and the recursion

`factor` handles the three smallest things. A `NUMBER` becomes a `Num` (this is where the raw text from §2.2.3 finally becomes a `float`). A `(` re-enters `expression` — the single recursive call that lets parentheses override precedence to any depth. A leading `-` becomes a `Unary`.

📄 [calc/parser.py lines 57–70](../sample_repo/calc/parser.py)

```python
# calc/parser.py : lines 57-70
    def factor(self) -> Node:
        tok = self.peek()
        if tok.kind == NUMBER:
            self.advance()
            return Num(float(tok.value))
        if tok.kind == LPAREN:
            self.advance()
            node = self.expression()
            self.expect(RPAREN)
            return node
        if tok.kind == MINUS:
            self.advance()
            return Unary("-", self.factor())
        raise SyntaxError(f"unexpected token {tok.kind}")
```

That call back up to `self.expression()` inside the parentheses case is the 'descent' in recursive descent: an arbitrarily nested `((1 + 2) * (3 + 4))` is handled by the same three methods calling one another. And `parse()` (§3.3 earlier, lines 38–41) finishes by demanding `EOF`, so trailing garbage like `1 + 2 )` is rejected rather than silently ignored.

## See it run

Parse two expressions and look at the shapes they produce:

In [1]:
import os, sys
# Find the bundled demo package (works when run from the calc-literate directory).
d = os.getcwd()
for _ in range(6):
    for cand in (os.path.join(d, 'sample_repo'), os.path.join(d, '..', 'sample_repo')):
        if os.path.isdir(os.path.join(cand, 'calc')):
            sys.path.insert(0, os.path.abspath(cand))
            break
    else:
        d = os.path.dirname(d); continue
    break
import calc
print('imported calc', calc.__version__)

imported calc 0.1.0


In [2]:
from calc.tokens import tokenize
from calc.parser import parse
for expr in ['1 + 2 * 3', '(1 + 2) * 3']:
    print(expr, '->', parse(tokenize(expr)))

1 + 2 * 3 -> BinOp(op='+', left=Num(value=1.0), right=BinOp(op='*', left=Num(value=2.0), right=Num(value=3.0)))
(1 + 2) * 3 -> BinOp(op='*', left=BinOp(op='+', left=Num(value=1.0), right=Num(value=2.0)), right=Num(value=3.0))


Notice how `(1 + 2) * 3` puts the **addition** deeper than the multiplication — the opposite nesting from `1 + 2 * 3` — which is precisely how the parentheses change the answer. The tree *is* the meaning.

## Design notes

- **Precedence by grammar vs. by table.** Recursive descent encodes precedence in the call structure. The common alternative — a precedence-climbing or Pratt parser driven by a table of binding powers — scales better to many operators, at the cost of being less transparent. For four operators, layered rules win on clarity.
- **No look-ahead beyond one token.** Every decision is made from `peek()` alone, which is what makes this grammar a clean `LL(1)`.

## Where this leads

The parser hands §4 a tree in which structure already means precedence. The evaluator can therefore be the simplest chapter in the book.